# `fast_match`: usage and timing

This notebook generates two simulated sky catalogues in memory, runs the three
matching modes, and reports wall-clock time. It does not write any catalogue
or result file.

`fast_match` takes NumPy coordinate arrays (and optional IDs) and returns a
pandas table of matched pairs.


In [1]:
import time

import numpy as np
import pandas as pd

from astrokit.toolbox.match import fast_match

rng = np.random.default_rng(20260822)
n_main = 20_000_000
n_counterpart = 14_000_000
n_background = 80_000

# Uniform 10 x 4 deg field; 70% of main sources have a 0.2 arcsec counterpart.
ra_1 = rng.uniform(150.0, 160.0, n_main)
dec_1 = rng.uniform(-2.0, 2.0, n_main)
id_1 = np.arange(n_main)

counterpart_index = rng.choice(n_main, n_counterpart, replace=False)
offset_arcsec = rng.normal(0.0, 0.2, size=(n_counterpart, 2))
ra_counterpart = (
    ra_1[counterpart_index]
    + offset_arcsec[:, 0] / (3600.0 * np.cos(np.deg2rad(dec_1[counterpart_index])))
)
dec_counterpart = dec_1[counterpart_index] + offset_arcsec[:, 1] / 3600.0

ra_2 = np.concatenate((ra_counterpart, rng.uniform(150.0, 160.0, n_background)))
dec_2 = np.concatenate((dec_counterpart, rng.uniform(-2.0, 2.0, n_background)))
id_2 = np.arange(n_counterpart + n_background)

ra_1.shape, ra_2.shape


((20000000,), (14080000,))

## Basic usage

Required inputs are 1D arrays: `ra_1`, `dec_1`, `ra_2`, `dec_2` (degrees).
`id_1` / `id_2` are optional; if omitted, positional indices `0 .. N-1` are
written into the result.

The returned pandas table always has columns
`id_1`, `ra_1`, `dec_1`, `id_2`, `ra_2`, `dec_2`, and `sep` (arcsec).


In [4]:
result = fast_match(
    ra_1, dec_1, ra_2, dec_2,
    radius_arcsec=1.0,
    id_1=id_1,
    id_2=id_2,
    mode='nearest',
)
print(f"Total: {len(result):,}")
result.head()


Total: 14,492,264


,id_1,ra_1,dec_1,id_2,ra_2,dec_2,sep
0,0,155.650961,-1.467190,9784246,155.650884,-1.467228,0.311374
1,3,152.460027,-0.520104,3747520,152.459986,-0.520158,0.242053
2,8,158.642505,-1.388862,11664762,158.642541,-1.388817,0.208216
3,9,156.615641,-1.305929,1069690,156.615630,-1.306005,0.277127
4,11,156.178930,1.113781,8611191,156.178938,1.113800,0.074620


## Performance test

Mode meanings and STILTS `tmatch2 find=` equivalents:

- `all` → `find=all`: every pair within the radius
- `nearest` → `find=best1`: one nearest neighbour per catalogue-1 source; catalogue-2 rows may be reused
- `best` → `find=best`: greedy one-to-one pairs ranked by separation

`workers=-1` uses all available CPU cores in SciPy's KD-tree query.


In [3]:
timings = []
for mode in ('all', 'nearest', 'best'):
    start = time.perf_counter()
    matched = fast_match(
        ra_1, dec_1, ra_2, dec_2,
        radius_arcsec=1.0,
        id_1=id_1,
        id_2=id_2,
        mode=mode,
        workers=-1,
    )
    elapsed = time.perf_counter() - start
    timings.append({'mode': mode, 'n_matches': len(matched), 'seconds': elapsed})

pd.DataFrame(timings).sort_values('mode').reset_index(drop=True)


,mode,n_matches,seconds
0,all,15707874,24.729027
1,best,14001301,39.439663
2,nearest,14492264,12.705464
